# Loading the dependencies

In [15]:
from datasets import load_dataset
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
import numpy as np

# Loading the data set and transforming for various models

Load the dataset from the tensorflow keras and split into the training and test data

In [16]:
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

Transforming the data for various Models MLP, CNN and Transformer

*   For MLP, normalize the images into a value between 0-255
*   For CNN, reshape the data to add a static height info



In [17]:
mlp_train_images = train_images / 255.0
mlp_test_images = test_images / 255.0

In [18]:
cnn_train_images = train_images.reshape((60000, 28, 28, 1)).astype('float32') / 255.0
cnn_test_images = test_images.reshape((10000, 28, 28, 1)).astype('float32') / 255.0

In [19]:
transformer_train_images = train_images / 255.0
transformer_test_images = test_images / 255.0

# MLP Model

Training for the MLP model using:


*   Optimizer = adam
*   loss function = sparse categorical crossentropy
*   Hidden layers size = 128, 64
*   Activation function = relu
*   Output layer size = 10
*   Epochs = 5



In [20]:
mlp_model = models.Sequential([
    layers.Flatten(input_shape=(28, 28)),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

mlp_model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

mlp_model.fit(mlp_train_images, train_labels, epochs=5)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.9296 - loss: 0.2408
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9690 - loss: 0.1012
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9779 - loss: 0.0702
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9825 - loss: 0.0545
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9863 - loss: 0.0432


Evaluation results of the MLP model


*   Model Accuracy = 97.14%



In [21]:
mlp_model.evaluate(mlp_test_images, test_labels)

313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9714 - loss: 0.0938


[0.09382349997758865, 0.9714000225067139]

# CNN Model

Training for the CNN model using:


*   Optimizer = adam
*   loss function = sparse categorical crossentropy
*   Filter layers size = 32, 64
*   Hidden layers size = 64
*   Activation function = relu
*   Output layer size = 10
*   Epochs = 5

In [22]:
cnn_model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])


cnn_model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

cnn_model.fit(cnn_train_images, train_labels, epochs=5, batch_size=64, validation_split=0.1)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - accuracy: 0.8975 - loss: 0.3379 - val_accuracy: 0.9825 - val_loss: 0.0564
Epoch 2/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9610 - loss: 0.1294 - val_accuracy: 0.9887 - val_loss: 0.0443
Epoch 3/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9719 - loss: 0.0967 - val_accuracy: 0.9890 - val_loss: 0.0380
Epoch 4/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9761 - loss: 0.0804 - val_accuracy: 0.9908 - val_loss: 0.0344
Epoch 5/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9793 - loss: 0.0702 - val_accuracy: 0.9912 - val_loss: 0.0346


Evaluation results of the MLP model


*   Model Accuracy = 98.78%



In [23]:
cnn_model.evaluate(cnn_test_images, test_labels)

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9878 - loss: 0.0327


[0.03268119692802429, 0.9878000020980835]

# Transformer

Training for the Transformer using:


*   Optimizer = adam
*   loss function = sparse categorical crossentropy
*   Filter layers size = 64 with kernel size = 7
*   Attention layer heads = 8
*   Hidden layers size = 64
*   Activation function = relu
*   Output layer size = 10
*   Epochs = 5

In [24]:
attention_layer = layers.MultiHeadAttention(num_heads=8, key_dim=64)

transformer_model = models.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(64, kernel_size=7, strides=7, padding="valid"),
    layers.Reshape((16, 64)),
    layers.Lambda(lambda x: attention_layer(x, x)),
    layers.LayerNormalization(epsilon=1e-6),
    layers.GlobalAveragePooling1D(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])


transformer_model.compile(optimizer='adam',
                          loss='sparse_categorical_crossentropy',
                          metrics=['accuracy'])

transformer_model.fit(transformer_train_images, train_labels, epochs=5, batch_size=64)

Epoch 1/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.5314 - loss: 1.3378
Epoch 2/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.6126 - loss: 1.1178
Epoch 3/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6311 - loss: 1.0661
Epoch 4/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6431 - loss: 1.0330
Epoch 5/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.6514 - loss: 1.0082


Evaluation results of the MLP model


*   Model Accuracy = 66%



In [25]:
transformer_model.evaluate(transformer_test_images, test_labels)

313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.6605 - loss: 0.9958


[0.9958454370498657, 0.6604999899864197]